In [ ]:
%pip install -q huggingface_hub pyarrow pandas matplotlib requests numpy

In [ ]:
# ── Parameters ────────────────────────────────────────────────────────────────
ASSET       = "BTCUSDT"              # uppercase asset symbol
LOOK_BACK   = 1440                   # number of past candles (includes last_candle)
LOOK_AHEAD  = 240                    # number of future candles
DATETIME    = "2025-12-12 20:00:00" # current_time: moment after last_candle closes (UTC)
WINDOW_MODE = "exclusive"            # "exclusive" | "inclusive"
NORM_MODE   = "clip"                 # "clip" | "tanh"
K           = 100                    # k=100  →  ratio 0.01 maps to 1.0

In [ ]:
import io, re, requests
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from huggingface_hub import HfApi
from datetime import datetime, timedelta, timezone

REPO    = "https://huggingface.co/datasets/payamdavaee/candles/resolve/main"
REPO_ID = "payamdavaee/candles"

def load_range(asset, start, end, columns=None):
    api      = HfApi()
    files    = api.list_repo_files(REPO_ID, repo_type="dataset")
    asset_lc = asset.lower()
    pattern  = re.compile(rf"^{asset_lc}/{asset_lc}-1m-(\d{{4}})-(\d{{2}})\.parquet$")
    start_ts = int(datetime.fromisoformat(start).replace(tzinfo=timezone.utc).timestamp() * 1000)
    end_ts   = int(datetime.fromisoformat(end  ).replace(tzinfo=timezone.utc).timestamp() * 1000) + 86_400_000
    frames   = []
    for f in sorted(files):
        m = pattern.match(f)
        if not m:
            continue
        resp = requests.get(f"{REPO}/{f}", timeout=60)
        resp.raise_for_status()
        tbl  = pq.read_table(io.BytesIO(resp.content), columns=columns or None)
        df   = tbl.to_pandas()
        ts   = df["ts"].astype("int64")
        df   = df[(ts >= start_ts) & (ts < end_ts)]
        if not df.empty:
            frames.append(df)
    if not frames:
        return pd.DataFrame()
    return pd.concat(frames).sort_values("ts").reset_index(drop=True)


# ── Derive window boundaries from DATETIME ────────────────────────────────────
current_time     = datetime.fromisoformat(DATETIME).replace(tzinfo=timezone.utc)
last_candle_time = current_time - timedelta(minutes=1)           # open time of last_candle
lb_start_time    = last_candle_time - timedelta(minutes=LOOK_BACK - 1)
lb_end_time      = last_candle_time
la_start_time    = current_time
la_end_time      = current_time + timedelta(minutes=LOOK_AHEAD - 1)

load_start = lb_start_time.strftime("%Y-%m-%d")
load_end   = (la_end_time + timedelta(days=1)).strftime("%Y-%m-%d")

print(f"Loading {ASSET} from {load_start} to {load_end} ...")
df_all = load_range(ASSET, load_start, load_end, columns=["ts", "vwap"])
print(f"Loaded {len(df_all):,} candles")

ts_int      = df_all["ts"].astype("int64")
lb_start_ms = int(lb_start_time.timestamp() * 1000)
lb_end_ms   = int(lb_end_time.timestamp()   * 1000)
la_start_ms = int(la_start_time.timestamp() * 1000)
la_end_ms   = int(la_end_time.timestamp()   * 1000)

df_lb = df_all[(ts_int >= lb_start_ms) & (ts_int <= lb_end_ms)].reset_index(drop=True)
df_la = df_all[(ts_int >= la_start_ms) & (ts_int <= la_end_ms)].reset_index(drop=True)

assert len(df_lb) == LOOK_BACK,  f"Expected {LOOK_BACK} look-back candles, got {len(df_lb)}"
assert len(df_la) == LOOK_AHEAD, f"Expected {LOOK_AHEAD} look-ahead candles, got {len(df_la)}"

print(f"Look-back  : {len(df_lb)} candles  ✓")
print(f"Look-ahead : {len(df_la)} candles  ✓")

In [ ]:
# ── Border datetimes ──────────────────────────────────────────────────────────
def fmt_ms(ts_ms):
    return pd.Timestamp(int(ts_ms), unit="ms", tz="UTC").strftime("%Y-%m-%d %H:%M:%S UTC")

lb_first_ts = df_lb["ts"].astype("int64").iloc[0]
lb_last_ts  = df_lb["ts"].astype("int64").iloc[-1]
la_first_ts = df_la["ts"].astype("int64").iloc[0]
la_last_ts  = df_la["ts"].astype("int64").iloc[-1]

print("─" * 52)
print(f"  current_time       :  {DATETIME} UTC")
print(f"  last_candle (open) :  {fmt_ms(lb_last_ts)}")
print("─" * 52)
print(f"  look-back  start   :  {fmt_ms(lb_first_ts)}")
print(f"  look-back  end     :  {fmt_ms(lb_last_ts)}")
print(f"  look-ahead start   :  {fmt_ms(la_first_ts)}")
print(f"  look-ahead end     :  {fmt_ms(la_last_ts)}")
print("─" * 52)

In [ ]:
# ── VWAP charts (side-by-side, proportional x-axis, shared y-axis) ────────────
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

def plot_vwap(ax, df, title, color, y_right=False):
    x     = np.arange(len(df))
    vwaps = df["vwap"].values
    ts_dt = pd.to_datetime(df["ts"].astype("int64"), unit="ms", utc=True)

    ax.plot(x, vwaps, color=color, linewidth=0.9)

    # X-axis ticks (~6 labels, HH:MM only)
    n     = len(df)
    step  = max(1, n // 6)
    ticks = x[::step]
    ax.set_xticks(ticks)
    ax.set_xticklabels(
        [ts_dt.iloc[i].strftime("%H:%M") for i in ticks],
        fontsize=6,
    )
    ax.set_xlim(-1, n)
    ax.set_title(title, fontsize=9)
    ax.tick_params(axis="y", labelsize=7)

    if y_right:
        ax.yaxis.tick_right()
        ax.yaxis.set_label_position("right")


# Shared price range across both windows
vwap_all = pd.concat([df_lb["vwap"], df_la["vwap"]])
y_min    = vwap_all.min()
y_max    = vwap_all.max()
y_margin = (y_max - y_min) * 0.05
y_lim    = (y_min - y_margin, y_max + y_margin)

fig = plt.figure(figsize=(14, 3))
gs  = gridspec.GridSpec(1, 2, width_ratios=[LOOK_BACK, LOOK_AHEAD], wspace=0.04)
ax_lb = fig.add_subplot(gs[0])
ax_la = fig.add_subplot(gs[1])

plot_vwap(ax_lb, df_lb, f"{ASSET} — Look-Back ({LOOK_BACK} candles)  [past]",   "#26a69a")
plot_vwap(ax_la, df_la, f"{ASSET} — Look-Ahead ({LOOK_AHEAD} candles)  [future]", "#ef5350", y_right=True)

ax_lb.set_ylim(*y_lim)
ax_la.set_ylim(*y_lim)

plt.suptitle(f"VWAP  |  current_time: {DATETIME} UTC", fontsize=10, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── Normalization ─────────────────────────────────────────────────────────────
# Price normalization  (idea_normalize_based_on_last_price_clip)
price_l = df_lb["vwap"].values[-1]   # last candle of look-back window — anchor

raw_lb = (df_lb["vwap"].values - price_l) / price_l
raw_la = (df_la["vwap"].values - price_l) / price_l

if NORM_MODE == "clip":
    scaled_lb = np.clip(K * raw_lb, -1.0, 1.0)
    scaled_la = np.clip(K * raw_la, -1.0, 1.0)
elif NORM_MODE == "tanh":
    scaled_lb = np.tanh(K * raw_lb)
    scaled_la = np.tanh(K * raw_la)
else:
    raise ValueError(f"Unknown NORM_MODE: {NORM_MODE!r}")

# Time normalization (idea_normalize_based_on_last_price_clip — time section)
# t_la starts at 1.0 because the first look-ahead candle opens at current_time
t    = np.arange(LOOK_BACK) / LOOK_BACK                          # [0, 1/1440, ..., 1439/1440]
t_la = (LOOK_BACK + np.arange(LOOK_AHEAD)) / LOOK_BACK           # [1440/1440, ..., 1679/1440]

print(f"price_l        = {price_l:.6f}")
print(f"k              = {K}   (ratio {1/K:.4f} → 1.0)")
print(f"t[0]           = {t[0]:.6f}   first look-back candle")
print(f"t[-1]          = {t[-1]:.6f}   last look-back candle (last_candle)")
print(f"current_time   = 1.000000   (t=1.0, not in array)")
print(f"t_la[0]        = {t_la[0]:.6f}   first look-ahead candle (opens at current_time)")
print(f"t_la[-1]       = {t_la[-1]:.6f}   last look-ahead candle")
print(f"scaled_lb      [{scaled_lb.min():.4f}, {scaled_lb.max():.4f}]")
print(f"scaled_la      [{scaled_la.min():.4f}, {scaled_la.max():.4f}]")

In [ ]:
# ── Normalized VWAP charts (side-by-side, proportional x-axis, shared y-axis) ─
fig = plt.figure(figsize=(14, 3))
gs  = gridspec.GridSpec(1, 2, width_ratios=[LOOK_BACK, LOOK_AHEAD], wspace=0.04)
ax_lb = fig.add_subplot(gs[0])
ax_la = fig.add_subplot(gs[1], sharey=ax_lb)   # guaranteed identical y-axis

# ── Look-back ─────────────────────────────────────────────────────────────────
ax_lb.plot(t, scaled_lb, color="#26a69a", linewidth=0.8)
ax_lb.axhline(0,  color="#aaaaaa", linewidth=0.5, linestyle="--")
ax_lb.axhline( 1, color="#ff9800", linewidth=0.4, linestyle=":")
ax_lb.axhline(-1, color="#ff9800", linewidth=0.4, linestyle=":")
ax_lb.set_ylim(-1.08, 1.08)
ax_lb.set_xlabel("normalized time  t = i / look_back", fontsize=8)
ax_lb.set_ylabel("scaled vwap", fontsize=8)
ax_lb.set_title(f"Look-Back — normalized VWAP  (k={K}, mode={NORM_MODE})", fontsize=9)
ax_lb.tick_params(labelsize=7)

# ── Look-ahead ────────────────────────────────────────────────────────────────
ax_la.plot(t_la, scaled_la, color="#ef5350", linewidth=0.8)
ax_la.axhline(0,  color="#aaaaaa", linewidth=0.5, linestyle="--")
ax_la.axhline( 1, color="#ff9800", linewidth=0.4, linestyle=":")
ax_la.axhline(-1, color="#ff9800", linewidth=0.4, linestyle=":")
ax_la.set_xlabel("normalized time  t = (look_back + j) / look_back", fontsize=8)
ax_la.yaxis.tick_right()
ax_la.yaxis.set_label_position("right")
ax_la.set_ylabel("scaled vwap", fontsize=8)
ax_la.set_title("Look-Ahead — normalized VWAP  (same price_l anchor)", fontsize=9)
ax_la.tick_params(labelsize=7)

plt.suptitle(
    f"Normalized VWAP  |  price_l={price_l:.4f}   k={K}   current_time={DATETIME} UTC",
    fontsize=10,
    y=1.01,
)
plt.tight_layout()
plt.show()